# NLP Data Preprocessing — Foundations 


In [1]:
import re
import string
from collections import Counter

import nltk
import numpy as np
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("nltk:", nltk.__version__)
print("torch:", torch.__version__)


nltk: 3.9.4
torch: 2.13.0


## 1. Why raw text can't go straight into a model

Every other module in this course treats the input as already numeric — pixels, feature columns. Text never starts that way. This whole notebook is the machinery between "a sentence a human typed" and "a tensor a model can multiply."

```text
Raw text -> Clean + normalize -> Tokenize -> Build vocabulary -> Encode to numbers -> Pad/truncate -> Model-ready tensor
```

**Think about it:** can we hand a raw string like `"OMG!!! This movie wasn't bad at all :) 10/10"` directly to Logistic Regression or an LSTM? *(No — every model we build multiplies numbers. A raw string isn't one.)*

In [2]:
raw_sentence = "OMG!!! This movie wasn't bad at all :) 10/10"

try:
    torch.tensor(raw_sentence)
except Exception as e:
    print(f"torch.tensor(raw_sentence) fails: {type(e).__name__}: {e}")

print("\nRaw sentence: ", raw_sentence)
print("Just lowercased:", raw_sentence.lower(), " <- still has punctuation, an emoticon, a contraction")


torch.tensor(raw_sentence) fails: TypeError: new(): invalid data type 'str'

Raw sentence:  OMG!!! This movie wasn't bad at all :) 10/10
Just lowercased: omg!!! this movie wasn't bad at all :) 10/10  <- still has punctuation, an emoticon, a contraction


## 2. Text cleaning: remove noise, not meaning

Cleaning strips out parts of the text that don't help the task — but "helps the task" is the key phrase. Nothing here is unconditionally garbage.

| Cleaning step | Before | After |
|---|---|---|
| Remove HTML | `<br>Great movie` | `Great movie` |
| Remove URL | `visit https://abc.com` | `visit` |
| Remove @mention | `thanks @raj` | `thanks` |
| Normalize spaces | `very    good` | `very good` |
| Remove stray symbols | `movie *** great` | `movie great` |

**The important caveat:** for sentiment analysis, `!!!` and ALL CAPS can signal intensity — don't strip everything just because you can. For topic classification, punctuation and URLs are usually safe to drop.

In [3]:
def strip_html(text):
    return re.sub(r"<[^>]+>", " ", text)

def strip_urls(text):
    return re.sub(r"http\S+|www\S+", " ", text)

def strip_mentions(text):
    return re.sub(r"@\w+", " ", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_text_basic(text):
    text = strip_html(text)
    text = strip_urls(text)
    text = strip_mentions(text)
    text = normalize_whitespace(text)
    return text

messy_examples = [
    "<br>This movie was AMAZING!!! Visit https://review.com now",
    "thanks @raj for the recommendation, loved it",
    "very    good   acting   but   slow   pacing",
]

for example in messy_examples:
    print(f"before: {example!r}")
    print(f"after:  {clean_text_basic(example)!r}\n")


before: '<br>This movie was AMAZING!!! Visit https://review.com now'
after:  'This movie was AMAZING!!! Visit now'

before: 'thanks @raj for the recommendation, loved it'
after:  'thanks for the recommendation, loved it'

before: 'very    good   acting   but   slow   pacing'
after:  'very good acting but slow pacing'



## 3. Lowercasing: usually free, occasionally wrong

Lowercasing collapses `Movie`, `movie`, `MOVIE` into one token, shrinking the vocabulary.

**Think about it:** is lowercasing ever a bad idea? Consider:

```text
"US"   (the country)          "us"   (the pronoun)
```

For Named Entity Recognition, casing is often the strongest signal that a word is a proper noun — lowercasing can hurt more than help there.

In [4]:
variants = ["Movie", "movie", "MOVIE"]
print("Before lowercasing:", variants)
print("After lowercasing: ", [v.lower() for v in variants], " <- all three collapse to one token\n")

sentence_a = "US economy grew this year"
sentence_b = "Can you help us with the return?"
print(f"{sentence_a!r} -> lowercased: {sentence_a.lower()!r}")
print(f"{sentence_b!r} -> lowercased: {sentence_b.lower()!r}")
print("Notice: 'US' (country) and 'us' (pronoun) are now the SAME token -- a real risk for NER.")


Before lowercasing: ['Movie', 'movie', 'MOVIE']
After lowercasing:  ['movie', 'movie', 'movie']  <- all three collapse to one token

'US economy grew this year' -> lowercased: 'us economy grew this year'
'Can you help us with the return?' -> lowercased: 'can you help us with the return?'
Notice: 'US' (country) and 'us' (pronoun) are now the SAME token -- a real risk for NER.


## 4. Contractions and negation: the step that protects meaning

This is the single most important cleaning decision in the whole pipeline, because it's the one most likely to silently flip a label.

```text
wasn't -> was not      don't -> do not      can't -> can not      I'm -> I am
```

**Think about it:** is "not bad" the same as "bad"? *(No — "not bad" usually reads as mildly positive.)* If "not" gets buried inside a contraction, or removed later as a stopword, the model loses the one word that reverses the sentence's meaning.

In [5]:
contractions = {
    "wasn't": "was not", "weren't": "were not", "isn't": "is not", "aren't": "are not",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "can't": "can not", "won't": "will not", "i'm": "i am",
}

def expand_contractions(text):
    for short_form, expanded in contractions.items():
        text = text.replace(short_form, expanded)
    return text

sentence = "The movie wasn't bad"
expanded = expand_contractions(sentence.lower())
print(f"before: {sentence!r}")
print(f"after:  {expanded!r}\n")

naive_stopwords = {"the", "was", "a", "an", "at", "all"}   # "not" is NOT protected here on purpose
naive_filtered = [w for w in expanded.split() if w not in naive_stopwords]
print("naive stopword removal (no protection):", naive_filtered, " <- 'not' survived only by luck here")

# now force the failure: add "not" to the stopword list, as an unwary student might
broken_stopwords = naive_stopwords | {"not"}
broken_filtered = [w for w in expanded.split() if w not in broken_stopwords]
print("if 'not' is treated as a stopword:      ", broken_filtered, " <- meaning has flipped!")


before: "The movie wasn't bad"
after:  'the movie was not bad'

naive stopword removal (no protection): ['movie', 'not', 'bad']  <- 'not' survived only by luck here
if 'not' is treated as a stopword:       ['movie', 'bad']  <- meaning has flipped!


## 5. Punctuation and numbers: keep, strip, or replace?

Both are "it depends" decisions.

**Punctuation** — `"Great."` vs `"Great!"` vs `"Great!!!"` plausibly carry different intensity. Strip it for simple topic classification; keep it as a token for sentiment/sarcasm work.

**Numbers** — `"10/10 movie"` has the number AS the sentiment signal; `"I watched this in 2021"` has a year that's probably irrelevant. Keep, replace with a placeholder, or remove — depending on whether the number carries task information.

In [6]:
def strip_punctuation(text):
    return re.sub(r"[^\w\s]", "", text)

def punctuation_as_tokens(text):
    return re.findall(r"\w+|[!?.]", text)

exclaim_sentence = "Great!!!"
print("strip punctuation:      ", strip_punctuation(exclaim_sentence))
print("keep punctuation tokens:", punctuation_as_tokens(exclaim_sentence))

rating_sentence = "10/10 movie, watched it in 2021"
placeholder_version = re.sub(r"\b\d+\b", "<NUM>", rating_sentence)
print("\noriginal:            ", rating_sentence)
print("numbers as placeholder:", placeholder_version, " <- loses whether 10/10 or 1/10")


strip punctuation:       Great
keep punctuation tokens: ['Great', '!', '!', '!']

original:             10/10 movie, watched it in 2021
numbers as placeholder: <NUM>/<NUM> movie, watched it in <NUM>  <- loses whether 10/10 or 1/10


## 6. Emoji and emoticon handling

In social media, reviews, and chat data, emoji frequently carry more sentiment than the words around them.

```text
"The food was cold :)"
```

Read the words alone and it's negative — the `:)` could mean sarcasm, politeness, or genuinely mixed feelings. Deleting the emoticon throws away a real signal; converting it to a word lets the rest of the pipeline use it.

In [7]:
emoticon_map = {
    ":)": " smile ", ":-)": " smile ",
    ":(": " sad ", ":-(": " sad ",
    "🙃": " upside_down_smile ",
    "❤️": " love ",
}

def convert_emoticons(text):
    for symbol, word in emoticon_map.items():
        text = text.replace(symbol, word)
    return normalize_whitespace(text)

sarcasm_example = "Great job... battery died in 2 hrs 🙃"
print(f"before: {sarcasm_example!r}")
print(f"after:  {convert_emoticons(sarcasm_example)!r}")

mixed_example = "The food was cold :)"
print(f"\nbefore: {mixed_example!r}")
print(f"after:  {convert_emoticons(mixed_example)!r}")


before: 'Great job... battery died in 2 hrs 🙃'
after:  'Great job... battery died in 2 hrs upside_down_smile'

before: 'The food was cold :)'
after:  'The food was cold smile'


## 7. Tokenization: deciding what the model calls "one unit"

Tokenization splits cleaned text into the pieces the rest of the pipeline treats as atomic — word, character, or subword.

| Granularity | Example | Used for |
|---|---|---|
| Word | `"I love NLP"` -> `["I","love","NLP"]` | classical ML, RNN/LSTM/GRU (this course) |
| Character | `"cat"` -> `["c","a","t"]` | spelling correction, languages without clear word boundaries |
| Subword | `"unhappiness"` -> `["un","happiness"]` | modern Transformers — handles rare/unseen words gracefully |

**Think about it:** why would a Transformer prefer subwords over whole words? *(A fixed word vocabulary can't cover every word that will ever appear; breaking rare words into familiar pieces means the model is never completely stuck on an unseen word.)*

In [8]:
from nltk.tokenize import word_tokenize

sentence = "I love NLP and it's fascinating!"

word_tokens = word_tokenize(sentence)
print("word tokens:     ", word_tokens)

char_tokens = list("cat")
print("character tokens:", char_tokens)

# a TOY subword split (real Transformers learn this with Byte-Pair Encoding / WordPiece --
# this just illustrates the idea of breaking a rare word into familiar pieces).
toy_subword_vocab = {"un", "happiness", "play", "ing", "cat", "s"}

def toy_subword_split(word, vocab):
    if word in vocab:
        return [word]
    for i in range(len(word) - 1, 0, -1):
        prefix, suffix = word[:i], word[i:]
        if prefix in vocab and suffix in vocab:
            return [prefix, suffix]
    return [word]  # fall back: whole word as its own "unknown" piece

for w in ["unhappiness", "playing", "cats"]:
    print(f"subword split of {w!r}: {toy_subword_split(w, toy_subword_vocab)}")


word tokens:      ['I', 'love', 'NLP', 'and', 'it', "'s", 'fascinating', '!']
character tokens: ['c', 'a', 't']
subword split of 'unhappiness': ['un', 'happiness']
subword split of 'playing': ['play', 'ing']
subword split of 'cats': ['cat', 's']


## 8. Stopword removal: optional, and dangerous if careless

Stopwords are very common words — `the, is, am, are, was, in, on, at, of` — that classical models sometimes drop.

**The trap:** `"not good"` with naive stopword removal can become `"good"` — the meaning inverts silently. The fix is a **protected set**: `keep token if (token not in stopwords) OR (token in protected_words)`.

**Modern practice:** deep learning and Transformer pipelines usually skip stopword removal entirely — the model can learn on its own which words matter.

In [9]:
from nltk.corpus import stopwords as nltk_stopwords

stop_set = set(nltk_stopwords.words("english"))
protected_words = {"not", "no", "never"}

def remove_stopwords_safe(tokens):
    return [t for t in tokens if (t not in stop_set) or (t in protected_words)]

tokens = "the movie was not good and the acting was bad".split()
print("tokens before:       ", tokens)
print("after safe removal:  ", remove_stopwords_safe(tokens), " <- 'not' survives on purpose")
print("'not' in nltk stopwords list?", "not" in stop_set)


tokens before:        ['the', 'movie', 'was', 'not', 'good', 'and', 'the', 'acting', 'was', 'bad']
after safe removal:   ['movie', 'not', 'good', 'acting', 'bad']  <- 'not' survives on purpose
'not' in nltk stopwords list? True


## 9. Stemming vs. lemmatization

Both reduce a word to a base form. **Stemming** chops endings by rule — fast, sometimes not even a real word. **Lemmatization** looks the word up using vocabulary and part-of-speech — slower, always a real dictionary word.

| Word | Stemming | Lemmatization |
|---|---|---|
| studies | studi | study |
| running | run | run |
| better | better | good |
| children | children | child |

In [10]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["studies", "running", "better", "children"]
pos_tags = ["n", "v", "a", "n"]  # noun / verb / adjective / noun -- lemmatizer needs a rough POS hint

print(f"{'word':<12}{'stem':<12}{'lemma':<12}")
for w, pos in zip(words, pos_tags):
    print(f"{w:<12}{stemmer.stem(w):<12}{lemmatizer.lemmatize(w, pos=pos):<12}")


word        stem        lemma       
studies     studi       study       
running     run         run         
better      better      good        
children    children    child       


## 10. Building the vocabulary

Once text is tokenized, we need one fixed master list mapping every known token to an integer ID.

```text
<pad> -> 0   <unk> -> 1   movie -> 2   good -> 3   bad -> 4
```

`<pad>` fills shorter sequences to a fixed batch length. `<unk>` is the catch-all for any word never seen during training.

**The rule that must not be broken:** build the vocabulary only from the training data. Never let the test set influence which words exist.

In [11]:
def build_vocab(tokenized_texts, min_freq=1):
    counter = Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {"<pad>": 0, "<unk>": 1}
    for token, count in counter.items():
        if count >= min_freq:
            vocab[token] = len(vocab)
    return vocab

def encode_tokens(tokens, vocab):
    return [vocab.get(t, vocab["<unk>"]) for t in tokens]

training_tokenized = [
    ["movie", "was", "good"],
    ["movie", "was", "bad"],
    ["excellent", "acting"],
]

vocab = build_vocab(training_tokenized, min_freq=1)
print("vocabulary:", vocab)

unseen_example = ["movie", "was", "fantabulous"]   # "fantabulous" was never seen in training
print(f"\nencoding {unseen_example}:", encode_tokens(unseen_example, vocab), " <- unknown word -> <unk> (id 1)")


vocabulary: {'<pad>': 0, '<unk>': 1, 'movie': 2, 'was': 3, 'good': 4, 'bad': 5, 'excellent': 6, 'acting': 7}

encoding ['movie', 'was', 'fantabulous']: [2, 3, 1]  <- unknown word -> <unk> (id 1)


## 11. From tokens to numbers: four ways to represent text

```text
Bag of Words:   text -> count vector           (order lost)
TF-IDF:         text -> weighted count vector  (order lost, common words downweighted)
Token IDs:      text -> sequence of integers   (order kept)
Embeddings:     token ID -> dense vector        (order kept, meaning captured)
```

**Think about it:** does Bag-of-Words tell "dog bites man" from "man bites dog" apart? *(No — identical word counts, opposite meaning. This is exactly the gap sequence models exist to close — see Module 9.)*

In [12]:
bow_vocab = ["movie", "was", "good", "bad"]

def bag_of_words(tokens, vocab):
    return [tokens.count(w) for w in vocab]

doc = "movie was good good".split()
print("document:", doc)
print("BoW vector:", bag_of_words(doc, bow_vocab))

sent_a = "dog bites man".split()
sent_b = "man bites dog".split()
shared_vocab = ["dog", "bites", "man"]
print(f"\nBoW('dog bites man') = {bag_of_words(sent_a, shared_vocab)}")
print(f"BoW('man bites dog') = {bag_of_words(sent_b, shared_vocab)}  <- identical vectors, opposite meaning")


document: ['movie', 'was', 'good', 'good']
BoW vector: [1, 1, 2, 0]

BoW('dog bites man') = [1, 1, 1]
BoW('man bites dog') = [1, 1, 1]  <- identical vectors, opposite meaning


In [13]:
import math

docs = ["good movie good", "good acting", "bad movie", "boring movie"]

def term_frequency(word, doc_tokens):
    return doc_tokens.count(word)

def inverse_doc_frequency(word, all_docs_tokens):
    N = len(all_docs_tokens)
    df = sum(1 for d in all_docs_tokens if word in d)
    return math.log(N / df) + 1

docs_tokens = [d.split() for d in docs]
tf = term_frequency("good", docs_tokens[0])
idf = inverse_doc_frequency("good", docs_tokens)
print(f"TF('good', D1)  = {tf}")
print(f"IDF('good')     = {idf:.2f}")
print(f"TF-IDF('good', D1) = {tf * idf:.2f}  (manual)")

vectorizer = TfidfVectorizer(smooth_idf=False, norm=None)
tfidf_matrix = vectorizer.fit_transform(docs)
good_idx = list(vectorizer.get_feature_names_out()).index("good")
print(f"TF-IDF('good', D1) = {tfidf_matrix[0, good_idx]:.2f}  (sklearn, unsmoothed, unnormalized -- should match)")


TF('good', D1)  = 2
IDF('good')     = 1.69
TF-IDF('good', D1) = 3.39  (manual)
TF-IDF('good', D1) = 3.39  (sklearn, unsmoothed, unnormalized -- should match)


In [14]:
def make_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

tokens = "not very good movie".split()
print("unigrams:", make_ngrams(tokens, 1))
print("bigrams: ", make_ngrams(tokens, 2))
print("trigrams:", make_ngrams(tokens, 3))

bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(["not very good movie"])
print("\nsklearn bigram features:", bigram_vectorizer.get_feature_names_out())


unigrams: ['not', 'very', 'good', 'movie']
bigrams:  ['not very', 'very good', 'good movie']
trigrams: ['not very good', 'very good movie']

sklearn bigram features: ['good' 'good movie' 'movie' 'not' 'not very' 'very' 'very good']


In [15]:
vocab_size = len(vocab)
embedding_dim = 8

embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

sample_ids = torch.tensor([[vocab["movie"], vocab["was"], vocab["good"], 0, 0]])  # (batch=1, seq_len=5)
embedded = embedding_layer(sample_ids)

print("token IDs shape:", tuple(sample_ids.shape), " <- (batch, seq_len)")
print("embedded shape: ", tuple(embedded.shape), " <- (batch, seq_len, embedding_dim)")
print("embedding for 'movie':", embedded[0, 0].detach().numpy().round(3))


token IDs shape: (1, 5)  <- (batch, seq_len)
embedded shape:  (1, 5, 8)  <- (batch, seq_len, embedding_dim)
embedding for 'movie': [ 1.642 -0.16  -0.497  0.44  -0.758  1.078  0.801  1.681]


## 12. Padding and truncation

Deep learning trains in batches, and a batch is a rectangular tensor — every sequence in it must be the same length. Real sentences are not.

```text
"good movie"                    -> 2 tokens
"the movie was very good"       -> 5 tokens
```

**Think about it:** for a product review, is the important sentiment more likely at the start or the end? *(Often the end — when that's true, keeping the LAST `T` tokens instead of the first `T` can work better. No universal rule — check what helps on your data.)*

In [16]:
def pad_sequence(seq, max_len, pad_value=0):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [pad_value] * (max_len - len(seq))

def truncate_keep_last(seq, max_len):
    if len(seq) <= max_len:
        return seq
    return seq[-max_len:]

short_seq = [3, 2]
long_seq = [3, 2, 5, 6, 4, 8, 9]

print("pad short_seq to length 5:", pad_sequence(short_seq, max_len=5))
print("pad long_seq to length 5: ", pad_sequence(long_seq, max_len=5), " <- truncated by default pad_sequence")
print("truncate long_seq, keep LAST 5:", truncate_keep_last(long_seq, max_len=5))


pad short_seq to length 5: [3, 2, 0, 0, 0]
pad long_seq to length 5:  [3, 2, 5, 6, 4]  <- truncated by default pad_sequence
truncate long_seq, keep LAST 5: [5, 6, 4, 8, 9]


## 13. Preprocessing depends on the model

There is no single correct pipeline — the right steps depend on what's consuming the output.

| Model | Typical preprocessing |
|---|---|
| Naive Bayes | clean, lowercase, tokenize, Bag-of-Words or TF-IDF |
| Logistic Regression | clean, lowercase, TF-IDF, n-grams |
| LSTM / GRU | clean, tokenize, build vocabulary, token IDs, pad |
| Transformer (BERT, GPT, ...) | use the model's **own pretrained tokenizer** — don't build your own |

**Important:** if you're fine-tuning a pretrained Transformer, never hand-roll your own tokenizer or vocabulary — the model was trained against one exact tokenizer and vocabulary.

## 14. Full worked pipeline: raw review to model-ready tensor

Now we put every piece together, on a small sentiment dataset, producing exactly the `X` (padded token IDs) and `y` (labels) a model would train on -- and, for comparison, the classical TF-IDF + Logistic Regression alternative, done without data leakage.

In [17]:
reviews = [
    ("OMG!!! This movie wasn't bad at all :) 10/10", "positive"),
    ("Worst film ever... I want my money back!!!", "negative"),
    ("The acting was good, but the story was boring.", "negative"),
    ("Absolutely loved it. Great music and great ending!", "positive"),
    ("Not good. Too slow and too long.", "negative"),
    ("A brilliant, emotional, and beautiful movie.", "positive"),
]

def clean_text_full(text):
    text = text.lower()
    text = expand_contractions(text)
    text = convert_emoticons(text)
    text = strip_urls(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = normalize_whitespace(text)
    return text

def preprocess_review(text):
    cleaned = clean_text_full(text)
    tokens = cleaned.split()
    tokens = remove_stopwords_safe(tokens)
    return tokens

processed = [(preprocess_review(text), label) for text, label in reviews]
for (tokens, label), (raw, _) in zip(processed, reviews):
    print(f"{raw!r}\n  -> {tokens}  [{label}]\n")

vocab_full = build_vocab([tokens for tokens, _ in processed], min_freq=1)
label_to_id = {"negative": 0, "positive": 1}
MAX_LEN = 10

X = [pad_sequence(encode_tokens(tokens, vocab_full), MAX_LEN, pad_value=vocab_full["<pad>"])
     for tokens, _ in processed]
y = [label_to_id[label] for _, label in processed]

X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

print("vocab size:", len(vocab_full))
print("X shape:", tuple(X_tensor.shape), " <- (num_reviews, MAX_LEN) -- ready for nn.Embedding, then Module 9's LSTM")
print("y shape:", tuple(y_tensor.shape))


"OMG!!! This movie wasn't bad at all :) 10/10"
  -> ['omg', 'movie', 'not', 'bad', 'smile', '10', '10']  [positive]

'Worst film ever... I want my money back!!!'
  -> ['worst', 'film', 'ever', 'want', 'money', 'back']  [negative]

'The acting was good, but the story was boring.'
  -> ['acting', 'good', 'story', 'boring']  [negative]

'Absolutely loved it. Great music and great ending!'
  -> ['absolutely', 'loved', 'great', 'music', 'great', 'ending']  [positive]

'Not good. Too slow and too long.'
  -> ['not', 'good', 'slow', 'long']  [negative]

'A brilliant, emotional, and beautiful movie.'
  -> ['brilliant', 'emotional', 'beautiful', 'movie']  [positive]

vocab size: 28
X shape: (6, 10)  <- (num_reviews, MAX_LEN) -- ready for nn.Embedding, then Module 9's LSTM
y shape: (6,)


In [18]:
texts = [r for r, _ in reviews]
labels = [l for _, l in reviews]

X_train_text, X_test_text, y_train_label, y_test_label = train_test_split(
    texts, labels, test_size=0.34, random_state=RANDOM_SEED, stratify=labels
)

classical_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))),
    ("classifier", LogisticRegression()),
])

# .fit() below fits BOTH the TfidfVectorizer and the classifier using ONLY X_train_text --
# the vectorizer has never seen X_test_text at this point.
classical_pipeline.fit(X_train_text, y_train_label)

test_accuracy = classical_pipeline.score(X_test_text, y_test_label)
print("held-out test accuracy:", test_accuracy, " <- with only 6 reviews total this number is not meaningful,")
print("                                     the point is the LEAK-FREE fit/transform order, not the score.")

custom_sentence = ["The movie was slow but the ending was excellent"]
prediction = classical_pipeline.predict(custom_sentence)
print(f"prediction for {custom_sentence[0]!r}: {prediction[0]}")


held-out test accuracy: 0.3333333333333333  <- with only 6 reviews total this number is not meaningful,
                                     the point is the LEAK-FREE fit/transform order, not the score.
prediction for 'The movie was slow but the ending was excellent': positive


## 15. Data leakage: the mistake that makes your model look better than it is

This is the most common silent bug in NLP preprocessing — it doesn't crash anything, it just quietly inflates your test score.

```mermaid
flowchart LR
    A["Raw data"] --> B["Train/test split"]
    B --> C["Train text"] --> D["Fit vocabulary / TF-IDF"]
    D --> E["Transform train"] --> F["Train model"]
    B --> G["Test text"] --> H["Transform using the SAME fitted vocab/TF-IDF"] --> I["Evaluate"]
```

**Teaching point, one sentence:** fit preprocessing objects (vocabulary, TF-IDF vectorizer, scaler — anything that "learns" from data) only on the training split, then reuse that fitted object, unchanged, to transform validation and test data.

In [19]:
leaky_vectorizer = TfidfVectorizer()
leaky_vectorizer.fit(texts)  # WRONG: fit on train + test combined
leaky_vocab = set(leaky_vectorizer.get_feature_names_out())

correct_vectorizer = TfidfVectorizer()
correct_vectorizer.fit(X_train_text)  # RIGHT: fit on training split only
correct_vocab = set(correct_vectorizer.get_feature_names_out())

leaked_words = leaky_vocab - correct_vocab
print(f"words the WRONG pipeline knows about that the RIGHT pipeline does not: {sorted(leaked_words)}")
print("Those words only appear in the held-out test reviews -- the wrong pipeline has already")
print("'seen' them before training, which is exactly what data leakage means.")


words the WRONG pipeline knows about that the RIGHT pipeline does not: ['back', 'beautiful', 'brilliant', 'emotional', 'ever', 'film', 'long', 'money', 'my', 'not', 'slow', 'too', 'want', 'worst']
Those words only appear in the held-out test reviews -- the wrong pipeline has already
'seen' them before training, which is exactly what data leakage means.
